<a href="https://colab.research.google.com/github/solsthecoder/NLP-LAB/blob/main/FA23_BDS_045_YASRAB_TALIB_Lab05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import math
import itertools
from collections import Counter

train = [
    "the cat sat on the mat",
    "the cat ate the fish",
    "the dog sat on the rug",
    "the dog ate the bone"
]

test = [
    "the cat sat on the rug",
    "the fish sat on the mat"
]

# ==========================================
# TASK 1: Tokenization and Vocabulary
# ==========================================
def tokenize(sentence):
    """Lowercases, splits on spaces, and adds <s> and </s> markers."""
    words = sentence.lower().split()
    return ["<s>"] + words + ["</s>"]

# Apply to training and test sentences
train_tokens = [tokenize(s) for s in train]
test_tokens = [tokenize(s) for s in test]

print("--- TASK 1 RESULTS ---")
print("Train Tokens:")
for t in train_tokens:
    print(t)
print("\nTest Tokens:")
for t in test_tokens:
    print(t)

# Total number of tokens in training corpus with and without markers
all_train_tokens_with_markers = [token for sent in train_tokens for token in sent]
# Without markers means excluding <s> and </s>
all_train_tokens_without_markers = [token for sent in train_tokens for token in sent if token not in ("<s>", "</s>")]

print(f"\nTotal tokens in training (with markers): {len(all_train_tokens_with_markers)}")
print(f"Total tokens in training (without markers): {len(all_train_tokens_without_markers)}")

def build_vocab(sentences_tokens):
    """Build vocabulary V: all unique words plus </s> (no <s>)."""
    vocab = set()
    for sent in sentences_tokens:
        for token in sent:
            if token != "<s>":
                vocab.add(token)
    return vocab

V_set = build_vocab(train_tokens)
V = sorted(list(V_set))
print(f"\nSize of Vocabulary (V): {len(V)}")
print(f"Sorted Vocabulary: {V}")

# ==========================================
# TASK 2: Generating N-grams
# ==========================================
def get_ngrams(tokens, n):
    """Returns a list of n-gram tuples from tokens."""
    if n > len(tokens):
        return [] # Does not crash when n > sentence length
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

print("\n--- TASK 2 RESULTS ---")
print("First training sentence unigrams:", get_ngrams(train_tokens[0], 1))
print("First training sentence bigrams: ", get_ngrams(train_tokens[0], 2))
print("First training sentence trigrams:", get_ngrams(train_tokens[0], 3))

# Verify m tokens give m - n + 1 n-grams for n = 1, 2, 3
for idx, tokens in enumerate(train_tokens + test_tokens):
    m = len(tokens)
    for n in (1, 2, 3):
        ngrams = get_ngrams(tokens, n)
        assert len(ngrams) == max(0, m - n + 1), f"Mismatch in sentence {idx}, n={n}"
print("Verification passed: m tokens give m - n + 1 N-grams for n=1, 2, 3.")

# Test n larger than sentence length
print("Test n larger than sentence length (n=10 for sentence of length 7):", get_ngrams(train_tokens[0], 10))


# ==========================================
# TASK 3: Counting N-grams
# ==========================================
unigram_counts = Counter()
bigram_counts = Counter()
trigram_counts = Counter()

for tokens in train_tokens:
    unigram_counts.update(get_ngrams(tokens, 1))
    bigram_counts.update(get_ngrams(tokens, 2))
    trigram_counts.update(get_ngrams(tokens, 3))

print("\n--- TASK 3 RESULTS ---")
print("Unigram Counts (sorted):", sorted(unigram_counts.items(), key=lambda x: x[1], reverse=True))
print("Bigram Counts (sorted):", sorted(bigram_counts.items(), key=lambda x: x[1], reverse=True))
print("Trigram Counts (sorted):", sorted(trigram_counts.items(), key=lambda x: x[1], reverse=True))

print("\nTop 5 Most Common Bigrams:")
for bg, count in bigram_counts.most_common(5):
    print(f"{bg}: {count}")

print(f"\nDistinct unigrams: {len(unigram_counts)}")
print(f"Distinct bigrams: {len(bigram_counts)}")
print(f"Distinct trigrams: {len(trigram_counts)}")

def count_of(ngram):
    """Returns count of an N-gram, or 0 if unseen."""
    if len(ngram) == 1:
        return unigram_counts[tuple(ngram) if isinstance(ngram, list) else (ngram,)]
    elif len(ngram) == 2:
        return bigram_counts[tuple(ngram)]
    elif len(ngram) == 3:
        return trigram_counts[tuple(ngram)]
    return 0

print(f"Count of seen bigram ('the', 'cat'): {count_of(('the', 'cat'))}")
print(f"Count of unseen bigram ('fish', 'dog'): {count_of(('fish', 'dog'))}")


# ==========================================
# TASK 4: Unigram Model
# ==========================================
N_train = len(all_train_tokens_with_markers) # Total tokens including markers

def unigram_prob(w):
    """MLE Unigram probability P(w) = C(w) / N"""
    # w can be string or 1-tuple
    w_tuple = (w,) if isinstance(w, str) else w
    return unigram_counts[w_tuple] / N_train

print("\n--- TASK 4 RESULTS ---")
print("Unigram Probabilities Table:")
prob_sum = 0
for word_tuple in sorted(unigram_counts.keys()):
    p = unigram_prob(word_tuple)
    prob_sum += p
    print(f"P({word_tuple[0]}) = {round(p, 4)}")

assert abs(prob_sum - 1.0) < 1e-5, "Unigram probabilities do not sum to 1!"
print("Assertion passed: Unigram probabilities sum to 1.0")

def unigram_sentence_prob(sentence_tokens):
    """Calculates product of unigram probabilities for sentence."""
    p = 1.0
    for token in sentence_tokens:
        p *= unigram_prob(token)
    return p

print(f"Unigram prob for Test Sentence 1: {round(unigram_sentence_prob(test_tokens[0]), 6)}")
print(f"Unigram prob for Test Sentence 2: {round(unigram_sentence_prob(test_tokens[1]), 6)}")

# Permutations test
words_in_test1 = test_tokens[0][1:-1] # Exclude markers
perm_probs = set(round(unigram_sentence_prob(["<s>"] + list(p) + ["</s>"]), 6) for p in itertools.permutations(words_in_test1))
print(f"Unique unigram probabilities across all permutations of test sentence 1 words: {perm_probs}")


# ==========================================
# TASK 5: Bigram Model (MLE)
# ==========================================
def bigram_prob(prev, w):
    """MLE Bigram probability P(w | prev) = C(prev, w) / C(prev)"""
    denom = unigram_counts[(prev,)]
    if denom == 0:
        return 0.0
    return bigram_counts[(prev, w)] / denom

def next_word_distribution(prev):
    """Returns a dictionary of next words and their MLE probabilities given prev."""
    dist = {}
    for word in V:  # Use V directly, no need for + ["</s>"]
        p = bigram_prob(prev, word)
        if p > 0:
            dist[word] = round(p, 4)
    return dist

print("\n--- TASK 5 RESULTS ---")
dist_the = next_word_distribution("the")
print("Next word distribution for 'the':", dist_the)
assert abs(sum(dist_the.values()) - 1.0) < 1e-5
print("Assertion passed: Next-word distribution for 'the' sums to 1.")

def bigram_sentence_prob(sentence_tokens):
    """Calculates bigram sentence probability using the chain rule."""
    p = 1.0
    ngrams = get_ngrams(sentence_tokens, 2)
    for prev, w in ngrams:
        p *= bigram_prob(prev, w)
    return p

print(f"Bigram sentence prob for Test Sentence 1: {round(bigram_sentence_prob(test_tokens[0]), 6)}")
print(f"Bigram sentence prob for Test Sentence 2: {round(bigram_sentence_prob(test_tokens[1]), 6)}")

def predict_next(prev):
    """Returns most probable next word (breaks ties alphabetically)."""
    candidates = V
    best_word = None
    max_p = -1.0
    # Sort alphabetically to handle tie breaks deterministically
    for word in sorted(candidates):
        p = bigram_prob(prev, word)
        if p > max_p:
            max_p = p
            best_word = word
    return best_word

print("Predict next for 'the':", predict_next("the"))
print("Predict next for 'cat':", predict_next("cat"))
print("Predict next for 'dog':", predict_next("dog"))
print("Predict next for 'sat':", predict_next("sat"))
print("Predict next for 'on':", predict_next("on"))


# ==========================================
# TASK 6: Trigram Model
# ==========================================
def trigram_prob(w1, w2, w3):
    """Trigram probability P(w3 | w1, w2) = C(w1, w2, w3) / C(w1, w2)"""
    denom = bigram_counts[(w1, w2)]
    if denom == 0:
        return 0.0
    return trigram_counts[(w1, w2, w3)] / denom

def trigram_sentence_prob(sentence_tokens):
    """Bigram model for first word given <s>, trigram model for subsequent words."""
    if len(sentence_tokens) < 3:
        return 0.0
    # First transition: P(w1 | <s>)
    p = bigram_prob(sentence_tokens[0], sentence_tokens[1])

    # Subsequent transitions: P(wi | wi-2, wi-1)
    trigrams = get_ngrams(sentence_tokens, 3)
    for w1, w2, w3 in trigrams:
        p *= trigram_prob(w1, w2, w3)
    return p

print("\n--- TASK 6 RESULTS ---")
print(f"Trigram sentence prob for Test Sentence 1: {round(trigram_sentence_prob(test_tokens[0]), 6)}")
print(f"Trigram sentence prob for Test Sentence 2: {round(trigram_sentence_prob(test_tokens[1]), 6)}")

print("\nModel N-gram Counts Table:")
print(f"Unigrams: {len(unigram_counts)}")
print(f"Bigrams:  {len(bigram_counts)}")
print(f"Trigrams: {len(trigram_counts)}")

def get_unseen_ngrams(sentence_tokens, n):
    ngrams = get_ngrams(sentence_tokens, n)
    return [ng for ng in ngrams if count_of(ng) == 0]

for idx, tokens in enumerate(test_tokens):
    unseen_bg = get_unseen_ngrams(tokens, 2)
    unseen_tg = get_unseen_ngrams(tokens, 3)
    print(f"\nTest Sentence {idx + 1} Unseen Bigrams: {unseen_bg}")
    print(f"Test Sentence {idx + 1} Unseen Trigrams: {unseen_tg}")


# ==========================================
# TASK 7: Zero-Probability Problem
# ==========================================
def unseen_bigrams(sentence_tokens):
    return [bg for bg in get_ngrams(sentence_tokens, 2) if bigram_counts[bg] == 0]

print("\n--- TASK 7 RESULTS ---")
for idx, tokens in enumerate(test_tokens):
    ubgs = unseen_bigrams(tokens)
    print(f"Test Sentence {idx+1} unseen bigrams and their MLE probabilities:")
    for bg in ubgs:
        p = bigram_prob(bg[0], bg[1])
        print(f"  P({bg[1]} | {bg[0]}) = {p}")

# Possible word pairs (words only, markers excluded)
vocab_words_only = list(V_set)
total_possible_pairs = len(vocab_words_only) * len(vocab_words_only)
occuring_pairs = sum(1 for bg in bigram_counts if bg[0] != "<s>" and bg[1] != "</s>")
unseen_pairs_count = total_possible_pairs - occuring_pairs
unseen_percentage = (unseen_pairs_count / total_possible_pairs) * 100
print(f"Percentage of unseen word-only pairs: {round(unseen_percentage, 2)}%")

def safe_log(p):
    """Returns -inf if p == 0, else ln(p)."""
    if p == 0:
        return float('-inf')
    return math.log(p)

# Log probability of second test sentence under MLE bigram
log_prob_test2 = sum(safe_log(bigram_prob(prev, w)) for prev, w in get_ngrams(test_tokens[1], 2))
print(f"Log probability of Test Sentence 2 (MLE Bigram): {log_prob_test2}")


# ==========================================
# TASK 8: Smoothing
# ==========================================
def bigram_prob_addone(prev, w):
    """Add-one smoothed bigram probability: (C(v, w) + 1) / (C(v) + V)"""
    c_vw = bigram_counts[(prev, w)]
    c_v = unigram_counts[(prev,)]
    return (c_vw + 1) / (c_v + len(V))

print("\n--- TASK 8 RESULTS ---")
print("Side-by-side comparison:")
print(f"P('sat' | 'fish'): MLE = {round(bigram_prob('fish', 'sat'), 4)}, Add-one = {round(bigram_prob_addone('fish', 'sat'), 4)}")
print(f"P('cat' | 'the'):  MLE = {round(bigram_prob('the', 'cat'), 4)}, Add-one  = {round(bigram_prob_addone('the', 'cat'), 4)}")

# Assert sum to 1 over all V words for add-one smoothing
# Fixed: Only test words that can actually be a previous word (exclude "</s>")
for prev_word in V + ["<s>"]:
    if prev_word == "</s>":
        continue  # </s> cannot be a previous word since sentences end with it

    s = sum(bigram_prob_addone(prev_word, w) for w in V)
    assert abs(s - 1.0) < 1e-5, f"Failed for '{prev_word}': sum is {s}"

print("Assertion passed: Add-one smoothed probabilities sum to 1 for all valid previous words.")

def bigram_prob_addk(prev, w, k):
    """Add-k smoothed bigram probability: (C(v, w) + k) / (C(v) + k * V)"""
    c_vw = bigram_counts[(prev, w)]
    c_v = unigram_counts[(prev,)]
    return (c_vw + k) / (c_v + k * len(V))

print("\nP('sat' | 'fish') for different k values:")
print(f"| k    | P('sat' | 'fish') |")
print(f"|---|---|")
for k_val in [1, 0.5, 0.1, 0.01]:
    p_val = bigram_prob_addk('fish', 'sat', k_val)
    print(f"| {k_val} | {round(p_val, 4)} |")


# ==========================================
# TASK 9: Perplexity and Model Comparison
# ==========================================
def perplexity(sentences_tokens, prob_fn):
    """Computes perplexity across sentences. Returns infinity if any prob is 0."""
    log_sum = 0.0
    total_words = 0
    for tokens in sentences_tokens:
        ngrams = get_ngrams(tokens, 2)
        for prev, w in ngrams:
            p = prob_fn(prev, w)
            if p == 0.0:
                return float('inf')
            log_sum += math.log(p)
        total_words += (len(tokens) - 1) # excluding <s>
    return math.exp(-log_sum / total_words)

# Unigram add-one probability function wrapper
def unigram_addone_prob_wrapper(prev, w):
    # P(w) = (C(w) + 1) / (N + V)
    return (unigram_counts[(w,)] + 1) / (N_train + len(V))

print("\n--- TASK 9 RESULTS ---")
pp_mle = perplexity(test_tokens, bigram_prob)
pp_addone = perplexity(test_tokens, bigram_prob_addone)
pp_uni_addone = perplexity(test_tokens, lambda prev, w: unigram_addone_prob_wrapper(prev, w))

print("Model Perplexity Comparison Table:")
print("| Model | Perplexity |")
print("|---|---|")
print(f"| Bigram MLE | {round(pp_mle, 4) if pp_mle != float('inf') else 'inf'} |")
print(f"| Bigram Add-One | {round(pp_addone, 4)} |")
print(f"| Unigram Add-One | {round(pp_uni_addone, 4)} |")

# Add-k perplexities
print("\nAdd-k Bigram Perplexities on Test Set:")
best_k = None
min_pp = float('inf')
for k in [1, 0.5, 0.1, 0.01]:
    pp = perplexity(test_tokens, lambda prev, w, kv=k: bigram_prob_addk(prev, w, kv))
    print(f"k = {k}: Perplexity = {round(pp, 4)}")
    if pp < min_pp:
        min_pp = pp
        best_k = k
print(f"Best k on test set: {best_k}")

# Train on first 3 sentences, tune on 4th sentence
train_subset = train[:3]
val_sentence = [train[3]]
train_subset_tokens = [tokenize(s) for s in train_subset]
val_tokens = [tokenize(s) for s in val_sentence]

# Recompute counts for subset
sub_unigram_counts = Counter()
sub_bigram_counts = Counter()
for tokens in train_subset_tokens:
    sub_unigram_counts.update(get_ngrams(tokens, 1))
    sub_bigram_counts.update(get_ngrams(tokens, 2))

def sub_bigram_prob_addk(prev, w, k):
    c_vw = sub_bigram_counts[(prev, w)]
    c_v = sub_unigram_counts[(prev,)]
    return (c_vw + k) / (c_v + k * len(V))

print("\nTuning k on Validation Sentence (Sentence 4):")
best_sub_k = None
min_sub_pp = float('inf')
for k in [1, 0.5, 0.1, 0.01, 0.001]:
    log_sum = 0.0
    total_words = 0
    for tokens in val_tokens:
        for prev, w in get_ngrams(tokens, 2):
            p = sub_bigram_prob_addk(prev, w, k)
            log_sum += math.log(p)
        total_words += (len(tokens) - 1)
    pp = math.exp(-log_sum / total_words)
    print(f"k = {k}: Validation Perplexity = {round(pp, 4)}")
    if pp < min_sub_pp:
        min_sub_pp = pp
        best_sub_k = k

print(f"Best k found via validation: {best_sub_k}")

--- TASK 1 RESULTS ---
Train Tokens:
['<s>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '</s>']
['<s>', 'the', 'cat', 'ate', 'the', 'fish', '</s>']
['<s>', 'the', 'dog', 'sat', 'on', 'the', 'rug', '</s>']
['<s>', 'the', 'dog', 'ate', 'the', 'bone', '</s>']

Test Tokens:
['<s>', 'the', 'cat', 'sat', 'on', 'the', 'rug', '</s>']
['<s>', 'the', 'fish', 'sat', 'on', 'the', 'mat', '</s>']

Total tokens in training (with markers): 30
Total tokens in training (without markers): 22

Size of Vocabulary (V): 11
Sorted Vocabulary: ['</s>', 'ate', 'bone', 'cat', 'dog', 'fish', 'mat', 'on', 'rug', 'sat', 'the']

--- TASK 2 RESULTS ---
First training sentence unigrams: [('<s>',), ('the',), ('cat',), ('sat',), ('on',), ('the',), ('mat',), ('</s>',)]
First training sentence bigrams:  [('<s>', 'the'), ('the', 'cat'), ('cat', 'sat'), ('sat', 'on'), ('on', 'the'), ('the', 'mat'), ('mat', '</s>')]
First training sentence trigrams: [('<s>', 'the', 'cat'), ('the', 'cat', 'sat'), ('cat', 'sat', 'on'), ('sat', '